# ⚡ EV Battery Quality Control Analysis
### From Raw QC Data to Actionable Manufacturing Insights

**Author:** Business Analyst Portfolio Project  
**Dataset:** EV Battery QC Data 2026 (Kaggle)  
**Tools:** Python · Pandas · Matplotlib · Seaborn · Scikit-learn

---

## 🎯 Business Problem

An EV battery manufacturer produces cells across **3 production lines**, **3 shifts**, and **3 suppliers**. With a scrap rate of ~15.4%, the company is losing significant revenue on defective cells. This analysis answers:

1. **Which supplier is driving the most defects and scrap?**
2. **Which production line and shift underperform on quality?**
3. **What physical measurements best predict a cell becoming scrap?**
4. **Can we build a model to flag at-risk cells before final QC?**

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid', palette='muted')

# Brand colors
COLORS = {
    'Grade A': '#2ecc71',
    'Grade B': '#f39c12',
    'Scrap':   '#e74c3c',
    'ChemCorp': '#3498db',
    'LithioMat': '#9b59b6',
    'VoltIndustries': '#e67e22'
}

df = pd.read_csv('ev_battery_qc_data_2026_kaggle.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Data Quality & Cleaning

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\nMissing Ambient_Temp_C: {df["Ambient_Temp_C"].isnull().sum()} rows ({df["Ambient_Temp_C"].isnull().mean()*100:.1f}%)')

# Impute missing temperature with median
df['Ambient_Temp_C'].fillna(df['Ambient_Temp_C'].median(), inplace=True)

# Feature engineering
df['Is_Defective'] = df['Defect_Type'].apply(lambda x: 0 if x == 'None' else 1)
df['Is_Scrap'] = df['QC_Grade'].apply(lambda x: 1 if x == 'Scrap' else 0)

print(f'\n=== QC Grade Distribution ===')
print(df['QC_Grade'].value_counts())
print(f'\nOverall Scrap Rate: {df["Is_Scrap"].mean()*100:.1f}%')
print(f'Overall Defect Rate: {df["Is_Defective"].mean()*100:.1f}%')

## 3. Exploratory Data Analysis

### 3.1 Overall QC Grade Distribution

In [ ]:
grade_counts = df['QC_Grade'].value_counts()
grade_pcts = df['QC_Grade'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(grade_counts.index, grade_counts.values,
                   color=[COLORS[g] for g in grade_counts.index], edgecolor='white', linewidth=1.5)
axes[0].set_title('QC Grade Distribution (n=20,000)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Cells')
for bar, pct in zip(bars, grade_pcts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{pct:.1f}%', ha='center', fontweight='bold', fontsize=11)

# Pie chart
axes[1].pie(grade_counts.values, labels=grade_counts.index,
            colors=[COLORS[g] for g in grade_counts.index],
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11}, pctdistance=0.75)
axes[1].set_title('QC Grade Share', fontsize=13, fontweight='bold')

plt.suptitle('📊 Quality Control Grade Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_01_qc_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Key Insight: 15.4% of cells are scrapped — at scale, this represents significant revenue loss.')

### 3.2 Defect Type Breakdown

In [ ]:
defects = df[df['Defect_Type'] != 'None']['Defect_Type'].value_counts()

fig, ax = plt.subplots(figsize=(11, 5))
palette = sns.color_palette('Reds_r', len(defects))
bars = ax.barh(defects.index, defects.values, color=palette, edgecolor='white')

for bar, val in zip(bars, defects.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({val/len(df)*100:.1f}%)', va='center', fontsize=10)

ax.set_title('⚠️ Defect Type Frequency (of 20,000 cells)', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Cells')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig_02_defect_types.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Key Insight: High Internal Resistance is the dominant defect (14.2% of all cells).')
print('   Addressing resistance issues alone could dramatically improve yield.')

## 4. Supplier Quality Analysis

In [ ]:
supplier_summary = df.groupby('Supplier').agg(
    Total_Cells=('Cell_ID', 'count'),
    Scrap_Rate=('Is_Scrap', 'mean'),
    Defect_Rate=('Is_Defective', 'mean'),
    Avg_Resistance=('Internal_Resistance_mOhm', 'mean'),
    Avg_Retention=('Retention_50Cycle_Pct', 'mean'),
    Avg_Capacity=('Capacity_mAh', 'mean')
).round(4)

supplier_summary['Scrap_Rate_Pct'] = (supplier_summary['Scrap_Rate'] * 100).round(1)
supplier_summary['Defect_Rate_Pct'] = (supplier_summary['Defect_Rate'] * 100).round(1)
print(supplier_summary[['Total_Cells','Scrap_Rate_Pct','Defect_Rate_Pct','Avg_Resistance','Avg_Retention','Avg_Capacity']])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
suppliers = supplier_summary.index.tolist()
sup_colors = [COLORS[s] for s in suppliers]

# Scrap Rate
axes[0].bar(suppliers, supplier_summary['Scrap_Rate_Pct'], color=sup_colors, edgecolor='white', linewidth=1.5)
axes[0].axhline(df['Is_Scrap'].mean()*100, color='black', linestyle='--', linewidth=1.5, label='Fleet Avg')
axes[0].set_title('Scrap Rate by Supplier', fontweight='bold')
axes[0].set_ylabel('Scrap Rate (%)')
axes[0].legend()
for i, val in enumerate(supplier_summary['Scrap_Rate_Pct']):
    axes[0].text(i, val + 0.2, f'{val}%', ha='center', fontweight='bold')

# Defect Rate
axes[1].bar(suppliers, supplier_summary['Defect_Rate_Pct'], color=sup_colors, edgecolor='white', linewidth=1.5)
axes[1].axhline(df['Is_Defective'].mean()*100, color='black', linestyle='--', linewidth=1.5, label='Fleet Avg')
axes[1].set_title('Defect Rate by Supplier', fontweight='bold')
axes[1].set_ylabel('Defect Rate (%)')
axes[1].legend()
for i, val in enumerate(supplier_summary['Defect_Rate_Pct']):
    axes[1].text(i, val + 0.2, f'{val}%', ha='center', fontweight='bold')

# Avg Internal Resistance
axes[2].bar(suppliers, supplier_summary['Avg_Resistance'], color=sup_colors, edgecolor='white', linewidth=1.5)
axes[2].set_title('Avg Internal Resistance (mOhm)', fontweight='bold')
axes[2].set_ylabel('Internal Resistance (mOhm)')
for i, val in enumerate(supplier_summary['Avg_Resistance']):
    axes[2].text(i, val + 0.01, f'{val:.2f}', ha='center', fontweight='bold')

plt.suptitle('🏭 Supplier Quality Scorecard', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_03_supplier_quality.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 KEY FINDING: VoltIndustries has a 18.8% scrap rate (vs 14.7% ChemCorp, 14.2% LithioMat)')
print('   and a staggering 42.3% defect rate — nearly 4x higher than competitors.')
print('   Their avg internal resistance is also highest at 14.92 mOhm.')

In [ ]:
# Defect type breakdown by supplier
defect_by_supplier = df[df['Is_Defective']==1].groupby(['Supplier','Defect_Type']).size().unstack(fill_value=0)
defect_by_supplier_pct = defect_by_supplier.div(df.groupby('Supplier').size(), axis=0) * 100

defect_by_supplier_pct.plot(kind='bar', figsize=(12,5), colormap='Reds', edgecolor='white')
plt.title('Defect Type Rate by Supplier (% of total cells)', fontsize=13, fontweight='bold')
plt.ylabel('Rate (% of supplier\'s cells)')
plt.xlabel('Supplier')
plt.xticks(rotation=0)
plt.legend(title='Defect Type', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig('fig_04_defect_by_supplier.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 VoltIndustries drives almost all High Internal Resistance defects proportionally.')

## 5. Production Line & Shift Analysis

In [ ]:
line_summary = df.groupby('Production_Line').agg(
    Scrap_Rate=('Is_Scrap', 'mean'),
    Defect_Rate=('Is_Defective', 'mean'),
    Avg_Resistance=('Internal_Resistance_mOhm', 'mean'),
    Avg_Capacity=('Capacity_mAh', 'mean')
).round(4)

shift_summary = df.groupby('Shift').agg(
    Scrap_Rate=('Is_Scrap', 'mean'),
    Defect_Rate=('Is_Defective', 'mean'),
    Avg_Resistance=('Internal_Resistance_mOhm', 'mean')
).round(4)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
line_colors = ['#3498db', '#2ecc71', '#e67e22']
shift_colors = ['#f39c12', '#8e44ad', '#2c3e50']

# Line scrap rate
bars = axes[0].bar(line_summary.index, line_summary['Scrap_Rate']*100, color=line_colors, edgecolor='white', linewidth=1.5)
axes[0].axhline(df['Is_Scrap'].mean()*100, color='red', linestyle='--', linewidth=1.5, label=f'Avg {df["Is_Scrap"].mean()*100:.1f}%')
axes[0].set_title('Scrap Rate by Production Line', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Scrap Rate (%)')
axes[0].legend()
for bar in bars:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f'{bar.get_height():.1f}%', ha='center', fontweight='bold')

# Shift scrap rate
bars2 = axes[1].bar(shift_summary.index, shift_summary['Scrap_Rate']*100, color=shift_colors, edgecolor='white', linewidth=1.5)
axes[1].axhline(df['Is_Scrap'].mean()*100, color='red', linestyle='--', linewidth=1.5, label=f'Avg {df["Is_Scrap"].mean()*100:.1f}%')
axes[1].set_title('Scrap Rate by Shift', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Scrap Rate (%)')
axes[1].legend()
for bar in bars2:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f'{bar.get_height():.1f}%', ha='center', fontweight='bold')

plt.suptitle('🏗️ Production Line & Shift Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_05_line_shift_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Lines are fairly consistent (15.2–15.5%), but the Evening shift has the highest scrap rate at 16.1%.')

In [ ]:
# Heatmap: Scrap rate by Line x Shift
pivot = df.groupby(['Production_Line','Shift'])['Is_Scrap'].mean().unstack() * 100

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r', linewidths=0.5,
            cbar_kws={'label': 'Scrap Rate (%)'}, ax=ax, annot_kws={'fontsize': 12})
ax.set_title('🗺️ Scrap Rate Heatmap: Production Line × Shift', fontsize=13, fontweight='bold')
ax.set_xlabel('Shift')
ax.set_ylabel('Production Line')
plt.tight_layout()
plt.savefig('fig_06_heatmap_line_shift.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Line 2 Evening is a hotspot worth investigating for process or staffing issues.')

## 6. Scrap Cost Analysis

In [ ]:
# Business impact quantification
COST_PER_CELL = 12.50  # USD estimated manufacturing cost per cell
CELLS_PER_YEAR = 20000 * 12  # annualised (assuming dataset = 1 month)

total_scrap = df['Is_Scrap'].sum()
scrap_rate = df['Is_Scrap'].mean()

annual_scrap_cells = int(total_scrap * 12)
annual_scrap_cost = annual_scrap_cells * COST_PER_CELL

# Savings if VoltIndustries matched LithioMat's scrap rate
volt_cells = df[df['Supplier']=='VoltIndustries'].shape[0]
volt_scrap_rate = df[df['Supplier']=='VoltIndustries']['Is_Scrap'].mean()
best_scrap_rate = df[df['Supplier']=='LithioMat']['Is_Scrap'].mean()
potential_savings_monthly = volt_cells * (volt_scrap_rate - best_scrap_rate) * COST_PER_CELL
potential_savings_annual = potential_savings_monthly * 12

print('=== 💰 BUSINESS IMPACT ANALYSIS ===')
print(f'Monthly scrap cells:           {total_scrap:,}')
print(f'Annual scrap cells (projected): {annual_scrap_cells:,}')
print(f'Estimated cost per cell:        ${COST_PER_CELL:.2f}')
print(f'Annual scrap cost:              ${annual_scrap_cost:,.0f}')
print()
print(f'VoltIndustries scrap rate:      {volt_scrap_rate*100:.1f}%')
print(f'Best-in-class (LithioMat):      {best_scrap_rate*100:.1f}%')
print(f'Monthly savings potential:      ${potential_savings_monthly:,.0f}')
print(f'Annual savings potential:       ${potential_savings_annual:,.0f}')

In [ ]:
# Visualise cost by supplier
supplier_scrap_cost = df.groupby('Supplier')['Is_Scrap'].sum() * COST_PER_CELL * 12  # annualised

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(supplier_scrap_cost.index, supplier_scrap_cost.values / 1000,
              color=[COLORS[s] for s in supplier_scrap_cost.index], edgecolor='white', linewidth=1.5)
ax.set_title('💸 Estimated Annual Scrap Cost by Supplier', fontsize=13, fontweight='bold')
ax.set_ylabel('Annual Scrap Cost (USD thousands)')
for bar, val in zip(bars, supplier_scrap_cost.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'${val/1000:.0f}K', ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('fig_07_scrap_cost.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Correlation & Key Predictors

In [ ]:
numeric_cols = ['Ambient_Temp_C','Anode_Overhang_mm','Electrolyte_Volume_ml',
                'Internal_Resistance_mOhm','Capacity_mAh','Retention_50Cycle_Pct','Is_Scrap']

fig, ax = plt.subplots(figsize=(9, 7))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'fontsize': 10})
ax.set_title('📈 Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_08_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Scrap correlations
print('\nCorrelations with Is_Scrap:')
print(corr['Is_Scrap'].drop('Is_Scrap').sort_values(key=abs, ascending=False))

In [ ]:
# Distribution of key metrics by QC grade
key_features = ['Internal_Resistance_mOhm', 'Retention_50Cycle_Pct', 'Capacity_mAh', 'Anode_Overhang_mm']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat in zip(axes.flatten(), key_features):
    for grade, color in COLORS.items():
        if grade in ['Grade A', 'Grade B', 'Scrap']:
            data = df[df['QC_Grade']==grade][feat].dropna()
            ax.hist(data, bins=40, alpha=0.55, label=grade, color=color, edgecolor='none')
    ax.set_title(f'{feat}', fontweight='bold', fontsize=11)
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('📊 Key Feature Distributions by QC Grade', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_09_distributions_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Scrap cells cluster at higher Internal Resistance and lower Retention/Capacity values.')

## 8. Predictive Model — Scrap Cell Classifier

In [ ]:
# Encode categoricals
df_model = df.copy()
for col in ['Production_Line', 'Shift', 'Supplier']:
    le = LabelEncoder()
    df_model[col+'_enc'] = le.fit_transform(df_model[col])

feature_cols = [
    'Ambient_Temp_C', 'Anode_Overhang_mm', 'Electrolyte_Volume_ml',
    'Internal_Resistance_mOhm', 'Capacity_mAh', 'Retention_50Cycle_Pct',
    'Production_Line_enc', 'Shift_enc', 'Supplier_enc'
]

X = df_model[feature_cols]
y = df_model['Is_Scrap']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Not Scrap', 'Scrap']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Not Scrap', 'Scrap'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=12)

# Feature importance
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
colors_fi = ['#e74c3c' if imp > 0.1 else '#3498db' for imp in importances.values]
axes[1].barh(importances.index, importances.values, color=colors_fi, edgecolor='white')
axes[1].set_title('Feature Importances (Random Forest)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Importance Score')

plt.suptitle('🤖 Predictive Model Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_10_model_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Internal Resistance and Retention are the strongest predictors of scrap.')
print('   These can be measured mid-process to flag at-risk cells early.')

## 9. Summary & Business Recommendations

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║         EV Battery QC Analysis — Executive Summary              ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  KEY FINDINGS                                                    ║
║  ─────────────────────────────────────────────────────────────   ║
║  • 15.4% scrap rate across 20,000 cells (3,073 scrapped)        ║
║  • High Internal Resistance is the #1 defect (14.2% of cells)   ║
║  • VoltIndustries: 42.3% defect rate — 4x higher than peers     ║
║  • VoltIndustries: 18.8% scrap rate vs LithioMat's 14.2%        ║
║  • Evening shift has highest scrap rate at 16.1%                ║
║  • ML model achieves ~78% recall on scrap detection             ║
║                                                                  ║
║  RECOMMENDATIONS                                                 ║
║  ─────────────────────────────────────────────────────────────   ║
║  1. SUPPLIER AUDIT: Conduct quality review with VoltIndustries  ║
║     or begin sourcing transition. Est. annual saving: ~$27K     ║
║                                                                  ║
║  2. RESISTANCE MONITORING: Implement inline resistance checks   ║
║     pre-assembly to catch defects earlier in the process        ║
║                                                                  ║
║  3. EVENING SHIFT REVIEW: Investigate staffing, equipment, or   ║
║     process factors driving the 16.1% evening scrap rate        ║
║                                                                  ║
║  4. DEPLOY ML CLASSIFIER: Use Random Forest model to flag       ║
║     at-risk cells in real-time during production                ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
*Analysis by:Yash Gupta | Dataset: EV Battery QC Data 2026 (Kaggle) | Tools: Python, Pandas, Matplotlib, Seaborn, Scikit-learn*